# Phase 0 — Model Candidate Comparison

Evaluates candidate decoder-only models against the selection criteria for the Emotion Engine:

| Criterion | Description |
|---|---|
| Hookable residual stream | `transformer_lens` or `nnsight` support |
| Licence | Permissive for academic use |
| VRAM (steered inference) | Fits RTX A5000 (24 GB) |
| Community mech-interp | Existing activation/steering work |

**Candidates**: GPT-2 medium · Qwen2.5-7B-Instruct · Llama-3.1-8B-Instruct · Mistral-7B-v0.3 · Gemma-2-9B-it

In [ ]:
import pandas as pd

candidates = [
    {
        "model": "GPT-2 medium",
        "hf_id": "gpt2-medium",
        "params_B": 0.345,
        "architecture": "GPT-2 decoder",
        "n_layers": 24,
        "hidden_dim": 1024,
        "licence": "MIT",
        "transformer_lens_support": True,
        "vram_bf16_GB": 0.7,       # ~700 MB
        "vram_int4_GB": None,
        "mech_interp_community": "★★★★★",   # canonical TL model
        "notes": "Canonical TransformerLens model; en-only; used for rapid iteration",
    },
    {
        "model": "Qwen2.5-7B-Instruct",
        "hf_id": "Qwen/Qwen2.5-7B-Instruct",
        "params_B": 7.6,
        "architecture": "Qwen2 decoder",
        "n_layers": 28,
        "hidden_dim": 3584,
        "licence": "Qwen Research (Apache-2.0 compatible for non-commercial)",
        "transformer_lens_support": True,   # via HookedTransformer / nnsight
        "vram_bf16_GB": 15.3,
        "vram_int4_GB": 5.1,
        "mech_interp_community": "★★★☆☆",
        "notes": "Fits 24 GB in bf16; nnsight hooks confirmed",
    },
    {
        "model": "Llama-3.1-8B-Instruct",
        "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
        "params_B": 8.0,
        "architecture": "Llama decoder",
        "n_layers": 32,
        "hidden_dim": 4096,
        "licence": "Llama 3 Community (non-commercial OK)",
        "transformer_lens_support": True,
        "vram_bf16_GB": 16.0,
        "vram_int4_GB": 5.5,
        "mech_interp_community": "★★★★☆",
        "notes": "Large mech-interp community; weaker ja/zh vs Qwen",
    },
    {
        "model": "Mistral-7B-v0.3",
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "params_B": 7.3,
        "architecture": "Mistral decoder",
        "n_layers": 32,
        "hidden_dim": 4096,
        "licence": "Apache-2.0",
        "transformer_lens_support": True,
        "vram_bf16_GB": 14.6,
        "vram_int4_GB": 5.0,
        "mech_interp_community": "★★★☆☆",
        "notes": "Fully open licence; en-centric; weaker ja/zh",
    },
    {
        "model": "Gemma-2-9B-it",
        "hf_id": "google/gemma-2-9b-it",
        "params_B": 9.2,
        "architecture": "Gemma-2 decoder",
        "n_layers": 42,
        "hidden_dim": 3584,
        "licence": "Gemma Terms of Use",
        "transformer_lens_support": True,
        "vram_bf16_GB": 18.4,
        "vram_int4_GB": 6.1,
        "mech_interp_community": "★★☆☆☆",
        "notes": "Strong reasoning; en-centric; 18 GB leaves little headroom for steering overhead",
    },
]

df = pd.DataFrame(candidates)
display_cols = [
    "model", "params_B", "n_layers", "hidden_dim",
    "licence",
    "transformer_lens_support", "vram_bf16_GB", "vram_int4_GB",
    "mech_interp_community",
]
df[display_cols].set_index("model")

,params_B,n_layers,hidden_dim,licence,transformer_lens_support,vram_bf16_GB,vram_int4_GB,mech_interp_community,recommended_role
model,,,,,,,,,
GPT-2 medium,0.345,24,1024,MIT,True,0.7,NaN,★★★★★,fast ablation
Qwen2.5-7B-Instruct,7.600,28,3584,Qwen Research (Apache-2.0 compatible for non-c...,True,15.3,5.1,★★★☆☆,PRIMARY
Llama-3.1-8B-Instruct,8.000,32,4096,Llama 3 Community (non-commercial OK),True,16.0,5.5,★★★★☆,en-only fallback
Mistral-7B-v0.3,7.300,32,4096,Apache-2.0,True,14.6,5.0,★★★☆☆,Apache-2.0 fallback
Gemma-2-9B-it,9.200,42,3584,Gemma Terms of Use,True,18.4,6.1,★★☆☆☆,out-of-scope (too large for A5000 bf16 margin)


## Decision Matrix

Score each model on the five criteria (0–2 per criterion, max 10).

In [7]:
import pandas as pd

# Scope: English-only — multilingual criterion dropped.
# Scores: 0 = does not meet, 1 = partially, 2 = fully meets  (max 8)
scores = {
    #                           hook  licence  vram  mech-interp
    "GPT-2 medium":           [  2,     2,      2,      2     ],
    "Qwen2.5-7B-Instruct":    [  2,     2,      2,      1     ],
    "Llama-3.1-8B-Instruct":  [  2,     2,      2,      2     ],
    "Mistral-7B-v0.3":        [  2,     2,      2,      1     ],
    "Gemma-2-9B-it":          [  2,     1,      1,      1     ],
}
criteria = ["hookable_residual", "licence", "vram_24GB", "mech_interp"]
score_df = pd.DataFrame(scores, index=criteria).T
score_df["TOTAL"] = score_df.sum(axis=1)
score_df.sort_values("TOTAL", ascending=False)


,hookable_residual,licence,vram_24GB,mech_interp,TOTAL
GPT-2 medium,2,2,2,2,8
Llama-3.1-8B-Instruct,2,2,2,2,8
Qwen2.5-7B-Instruct,2,2,2,1,7
Mistral-7B-v0.3,2,2,2,1,7
Gemma-2-9B-it,2,1,1,1,5


## VRAM Budget Breakdown

RTX A5000 = **24 GB**. During steered inference we need:

| Item | Estimate |
|---|---|
| Model weights (Qwen2.5-7B bf16) | ~15.3 GB |
| KV-cache (256 gen tokens, bs=1) | ~0.8 GB |
| Activation buffer (4 layers × 3584 × batch) | ~0.1 GB |
| Steering vector(s) + misc | ~0.1 GB |
| **Total** | **~16.3 GB** ✓ (7.7 GB headroom) |

With `bitsandbytes` int-4 quantisation, the model drops to ~5 GB, enabling larger batch sizes for dataset-scale activation collection.

In [ ]:
import plotly.graph_objects as go

models = ["GPT-2 medium", "Qwen2.5-7B\n(bf16)", "Qwen2.5-7B\n(int4)",
          "Llama-3.1-8B\n(bf16)", "Mistral-7B\n(bf16)", "Gemma-2-9B\n(bf16)"]
vram   = [0.7,             15.3,                5.1,
          16.0,                14.6,               18.4]
colors = ["steelblue", "#2ca02c", "#98df8a", "orange", "slategray", "salmon"]

fig = go.Figure(go.Bar(x=models, y=vram, marker_color=colors,
                       text=[f"{v} GB" for v in vram], textposition="outside"))
fig.add_hline(y=24, line_dash="dash", line_color="red",
              annotation_text="A5000 limit", annotation_position="right")
fig.update_layout(title="VRAM usage per model (bf16 / int-4 where applicable)",
                  yaxis_title="VRAM (GB)", yaxis_range=[0, 26],
                  template="plotly_white", width=800)
fig.show()

## Hook Smoke-Test (GPT-2 medium via TransformerLens)

Confirms that we can intercept residual-stream activations before committing to the 7B model download.

In [6]:
# Run the dedicated smoke-test script (also usable in CI)
import subprocess, sys

result = subprocess.run(
    [sys.executable, "../tests/test_hook_smoke.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

KeyboardInterrupt: 

## Decision

**Scope change**: English-only. Multilingual criterion dropped.

With multilingual removed, Llama-3.1-8B-Instruct and GPT-2 medium both achieve the maximum score (8/8). Llama is selected as primary because it is the dominant architecture in the mech-interp / CAA literature — most published steering experiments (including the original CAA paper) target Llama-family models, which means method references, open-source steering implementations, and activation datasets transfer directly.

| Role | Model | Rationale |
|---|---|---|
| **PRIMARY** | **Llama-3.1-8B-Instruct** | 8/10; largest en mech-interp community; CAA literature directly targets Llama; fits A5000 in bf16 (16 GB, 8 GB headroom) |
| **Fast ablation** | **GPT-2 medium** | Canonical TL model; near-instant iteration; useful for hyperparameter sweeps |

Update `configs/model.yaml → active: llama` to use Llama-3.1-8B-Instruct in all downstream phases.  
Switch to `active: gpt2` for rapid ablation runs.
